# Alpha 因子评价

读取中性化 Alpha 暴露和 `return_t2`，在研究期非 ST 股票中计算每日 Rank IC，并逐因子汇总。2025 年作为回测期，不参与本步骤的因子筛选。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。

评价信号日截止至 2024-12-27，其收益最晚于 2024-12-31 开盘实现；排除年末两个会用到 2025 年行情的信号日。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

EVA_START = pd.Timestamp('20230101')
# Last signal whose t+2 opening return is realized by 2024-12-31.
EVA_END = pd.Timestamp('20241227')

ALPHA = PROJECT_ROOT / "src/factor_exposure/alpha_factor/alpha_factor_exposure.parquet"
RETURN_T2 = PROJECT_ROOT / "src/return_t2/return_t2.parquet"
PANEL = PROJECT_ROOT / "data/processed/panel.parquet"

## 读取输入数据

Alpha 文件保存中性化后的 33 个暴露；面板提供 ST 状态，收益表提供评价标签。


In [ ]:
alpha = pd.read_parquet(ALPHA)
return_t2 = pd.read_parquet(RETURN_T2)
panel = pd.read_parquet(PANEL)

## 对齐评价样本

按日期和股票合并 ST 状态、收益标签及 Alpha 暴露，再筛选研究期内的非 ST 记录。每个因子的有效配对在计算相关性时单独确定。


In [ ]:
evaluation = pd.merge(left=panel[['ts_code', 'trade_date', 'ST']], right=return_t2, on=['trade_date','ts_code'], how='left')

In [ ]:
evaluation = pd.merge(left=evaluation, right=alpha, on=['trade_date','ts_code'], how='left')

In [ ]:
evaluation = evaluation[evaluation['trade_date'].between(EVA_START, EVA_END) & (evaluation['ST'] == False)]

## 预览样本

查看字段和筛选后的数据规模，确认日期、股票和标签已经对齐。


In [ ]:
evaluation.head()

In [ ]:
evaluation.shape

## 逐日计算 Rank IC

对每个交易日、每个 Alpha，计算暴露与 `return_t2` 的 Spearman 相关系数。相关性使用两列同时有效的记录，结果保存在 `rank_ic` 中。

正值表示高暴露股票的后续收益排序更高；负值表示相反。


In [ ]:
daily_ic = []

In [ ]:
for date, group in evaluation.groupby('trade_date'):
    for i in range(4, len(group.columns)):
        rank_ic = group.iloc[:,i].corr(
                group["return_t2"],
                method="spearman",
            )
        
        daily_ic.append({
            'alpha': group.columns[i],
            "trade_date": date,
            "rank_ic": rank_ic,
        })

In [ ]:
rank_ic = pd.DataFrame(daily_ic)

## 汇总评价结果

剔除无效 Rank IC 后，统计有效交易日数、日均 IC、日 IC 样本标准差、ICIR 和正 IC 比例。ICIR 为均值除以标准差，不年化；`n_days` 是因子有效天数。


In [ ]:
for alpha, group in rank_ic.groupby('alpha'):
    group = group.loc[np.isfinite(group['rank_ic'])].copy()
    mean_ic = group['rank_ic'].mean()
    std_ic = group['rank_ic'].std(ddof=1)
    summarize = {
        'alpha': alpha,
        "n_days": group["trade_date"].nunique(),
        "mean_ic": mean_ic.round(4),
        "std_ic": std_ic.round(4),
        "icir": (mean_ic / std_ic).round(4),
        "positive_ic_ratio": (group['rank_ic'] > 0).mean().round(4)
        }
    print('========================================')
    print(pd.Series(summarize))

## 后续研究因子

本版后续收益预测与组合研究采用以下 8 个 Alpha，与因子收益回归和股票预测中的配置一致：

| 因子 | 进入收益模型的方向 |
| --- | --- |
| `alpha42`、`alpha1`、`alpha132`、`alpha12` | 保持中性化暴露的原方向 |
| `alpha80`、`alpha112`、`alpha177`、`alpha110` | 中性化暴露乘以 `-1` |

本 notebook 按暴露的原方向评价；方向转换在后续模型中执行，不回写 Alpha 暴露文件。负向因子通过反向使用进入模型，不因负 Rank IC 直接剔除。

这 8 个 Alpha 与 `size`、`mom`、`vol1`、`vol2`、`liq1`、`liq2`、`beta`、`value` 八个风格因子共同估计收益。其余 25 个候选保留评价结果，暂不纳入本版收益模型。

下一步运行 [factor_return.ipynb](../factor_return/factor_return.ipynb)，再运行 [stock_return.ipynb](../stock_return/stock_return.ipynb)。


## 报告图表：因子评价

生成33个候选的原方向平均 Rank IC，以及八个入选因子的研究期日截面暴露相关性均值。相关性逐日按有效配对计算后取均值，不混合不同日期直接计算。图片保存到 `src/figures/`。


In [ ]:
import matplotlib.pyplot as plt

FIGURE_DIR = PROJECT_ROOT / 'src/figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})

# Raw directions: selected factors are highlighted, not automatically reselected.
report_alpha_cols = ['alpha42', 'alpha80', 'alpha112', 'alpha177',
                     'alpha1', 'alpha110', 'alpha132', 'alpha12']
report_reverse_cols = ['alpha80', 'alpha112', 'alpha177', 'alpha110']
report_ic = rank_ic.pivot(index='trade_date', columns='alpha', values='rank_ic').sort_index()
report_ic_mean = report_ic.mean().sort_values()
fig, ax = plt.subplots(figsize=(9, 11), layout='constrained')
ax.barh(report_ic_mean.index, report_ic_mean,
        color=['#2563eb' if col in report_alpha_cols else '#cbd5e1'
               for col in report_ic_mean.index])
ax.axvline(0, color='#475569', linewidth=.8)
ax.set(title='Research-period mean Rank IC: all 33 candidates', xlabel='Mean Rank IC (raw direction)')
ax.grid(axis='x', alpha=.2)
fig.savefig(FIGURE_DIR / 'evaluation_mean_rank_ic.png', dpi=200)
plt.show()

# Average daily cross-sectional Spearman correlations, not pooled correlations.
report_exposures = evaluation[['trade_date'] + report_alpha_cols].copy()
report_exposures[report_reverse_cols] *= -1
report_correlations = (
    report_exposures.groupby('trade_date')[report_alpha_cols]
    .corr(method='spearman').groupby(level=1).mean()
    .reindex(index=report_alpha_cols, columns=report_alpha_cols)
)
fig, ax = plt.subplots(figsize=(9, 7), layout='constrained')
heat = ax.imshow(report_correlations, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(8), report_alpha_cols, rotation=45, ha='right')
ax.set_yticks(range(8), report_alpha_cols)
for row in range(8):
    for col in range(8):
        value = report_correlations.iloc[row, col]
        ax.text(col, row, f'{value:.2f}', ha='center', va='center',
                color='white' if abs(value) > .65 else 'black')
ax.set_title('Mean daily exposure rank correlation (model directions)')
fig.colorbar(heat, ax=ax, shrink=.8)
fig.savefig(FIGURE_DIR / 'evaluation_exposure_correlation.png', dpi=200)
plt.show()
